# Session 3 — Pixel Losses and Honest Scores 🧪

**Time:** 35–40 minutes  
**Goal:** score a segmentation prediction while excluding ignored border pixels.

You will work with tiny masks where every count is visible, then use the same accumulation pattern used for a validation set.

## 0. Setup

This lab has no download and no training step. Run the cells in order.

In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(7)
IGNORE_INDEX = 255
CLASS_NAMES = ['background', 'pet']
print('PyTorch:', torch.__version__)

PyTorch: 2.11.0+cpu


## 1. From logits to a mask (5 min)

A model gives two **logits** per pixel: one for background and one for pet. Softmax turns each pair into probabilities that sum to one; `argmax` selects the larger probability.

In [2]:
# Shape: [batch, classes, height, width]
logits = torch.tensor([[[[3.0, -1.0], [0.2, 1.2]],
                        [[-1.0, 3.0], [1.5, 0.3]]]])
probabilities = logits.softmax(dim=1)
prediction = probabilities.argmax(dim=1)

print('logits shape:       ', tuple(logits.shape))
print('probability sums:\n', probabilities.sum(dim=1))
print('prediction shape:   ', tuple(prediction.shape))
print('predicted labels:\n', prediction[0])
assert torch.allclose(probabilities.sum(dim=1), torch.ones_like(probabilities[:, 0]))
assert prediction.shape == (1, 2, 2)

logits shape:        (1, 2, 2, 2)
probability sums:
 tensor([[[1., 1.],
         [1., 1.]]])
prediction shape:    (1, 2, 2)
predicted labels:
 tensor([[0, 1],
        [1, 0]])


## 2. Ignored borders do not vote (6 min)

Oxford-IIIT Pet trimaps use `255` for an ambiguous border. It is neither background nor pet, so it must be excluded from both the loss and every metric.

In [3]:
targets = torch.tensor([[[0, 1], [IGNORE_INDEX, 0]]])
valid = targets != IGNORE_INDEX
loss = F.cross_entropy(logits, targets, ignore_index=IGNORE_INDEX)

print('valid pixels:', int(valid.sum()), 'of', targets.numel())
print('valid mask:\n', valid[0])
print(f'cross-entropy on valid pixels only: {loss.item():.3f}')
assert int(valid.sum()) == 3

valid pixels: 3 of 4
valid mask:
 tensor([[ True,  True],
        [False,  True]])
cross-entropy on valid pixels only: 0.126


## 3. Exercise E1 — accumulate only valid pixels (10 min)

Complete the helper below. It must update a 2×2 confusion matrix whose **rows are true classes** and **columns are predicted classes**. Ignore every target equal to `255`.

In [6]:
# TODO: update the confusion matrix using only target != IGNORE_INDEX pixels
def update_confusion(confusion, target, predicted, ignore_index=IGNORE_INDEX):
    valid = target != ignore_index
    for truth, guess in zip(target[valid].reshape(-1), predicted[valid].reshape(-1)):
        confusion[int(truth), int(guess)] += 1
    return confusion


def scores_from_confusion(confusion):
    total = confusion.sum().item()
    accuracy = confusion.diag().sum().item() / total

    ious = []
    for class_id in range(confusion.shape[0]):
        tp = confusion[class_id, class_id].item()
        fp = confusion[:, class_id].sum().item() - tp
        fn = confusion[class_id, :].sum().item() - tp
        union = tp + fp + fn
        ious.append(float("nan") if union == 0 else tp / union)

    return accuracy, torch.tensor(ious)


In [7]:
# Two small saved predictions: accumulate counts first, then score once.
batch_targets = [
    torch.tensor([[0, 1], [1, IGNORE_INDEX]]),
    torch.tensor([[0, 0], [1, 1]]),
]
batch_predictions = [
    torch.tensor([[0, 1], [0, 1]]),
    torch.tensor([[0, 1], [1, 0]]),
]

confusion = torch.zeros((2, 2), dtype=torch.int64)
for target, predicted in zip(batch_targets, batch_predictions):
    confusion = update_confusion(confusion, target, predicted)

accuracy, class_iou = scores_from_confusion(confusion)
print('rows=true, columns=predicted')
print(confusion)
print(f'dataset pixel accuracy: {accuracy:.3f}')
print(f'background IoU: {class_iou[0]:.3f}; pet IoU: {class_iou[1]:.3f}')
assert confusion.sum().item() == 7  # one of eight cells was ignored
assert torch.all((class_iou >= 0) & (class_iou <= 1))

rows=true, columns=predicted
tensor([[2, 1],
        [2, 2]])
dataset pixel accuracy: 0.571
background IoU: 0.400; pet IoU: 0.400


**Why accumulate?** Averaging IoU separately for each image gives every image equal weight, even if one has many more valid pixels. A validation report should state its aggregation rule; here we total counts over the dataset before calculating the score.

## 4. Exercise E2 — the worked 4×4 pet IoU (8 min)

For the pet class, find `TP`, `FP`, and `FN`. The ignored lower-right cell must disappear before counting. The expected result is $3/(3+1+1)=0.60$.

In [8]:
target_4x4 = torch.tensor([
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, IGNORE_INDEX],
])
predicted_4x4 = torch.tensor([
    [1, 1, 0, 0],
    [1, 0, 1, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 1],
])

In [9]:
# TODO: calculate the pet TP, FP, FN, and IoU while excluding ignored labels

valid = target_4x4 != IGNORE_INDEX

truth_pet = (target_4x4 == 1) & valid
pred_pet = (predicted_4x4 == 1) & valid

tp = int((truth_pet & pred_pet).sum())
fp = int((~truth_pet & pred_pet & valid).sum())
fn = int((truth_pet & ~pred_pet & valid).sum())

pet_iou = tp / (tp + fp + fn)

print(f'TP={tp}, FP={fp}, FN={fn}, pet IoU={pet_iou:.2f}')

assert (tp, fp, fn) == (3, 1, 1)
assert pet_iou == 0.60


TP=3, FP=1, FN=1, pet IoU=0.60


## 5. Accuracy can hide a bad foreground mask (5 min)

Suppose a 10×10 image contains only four pet pixels. A model predicts every pixel as background. It is correct on 96 of 100 pixels, but it finds none of the pet.

In [10]:
rare_target = torch.zeros((10, 10), dtype=torch.int64)
rare_target[:2, :2] = 1
all_background = torch.zeros_like(rare_target)

rare_confusion = torch.zeros((2, 2), dtype=torch.int64)
rare_confusion = update_confusion(rare_confusion, rare_target, all_background)
rare_accuracy, rare_iou = scores_from_confusion(rare_confusion)
print(f'pixel accuracy: {rare_accuracy:.2%}')
print(f'pet IoU: {rare_iou[1]:.2f}')
assert rare_accuracy == 0.96 and rare_iou[1] == 0.0

pixel accuracy: 96.00%
pet IoU: 0.00


> TODO: In one or two sentences, explain why the 96% pixel accuracy above is misleading.


Aunque el modelo obtiene una precisión del 96%, solo predice la clase de fondo y no detecta ningún píxel de la mascota. Esto demuestra que la pixel accuracy puede ser alta cuando una clase domina la imagen, mientras que el IoU revela que el rendimiento sobre la clase de interés es deficiente.

## Checkpoint

Before leaving, confirm that you can explain: (1) why `255` is excluded, (2) why the 4×4 pet IoU is 0.60, and (3) why a dataset metric starts from accumulated counts. Next session you will use these metrics to evaluate a pretrained segmentation model.

1.- El valor 255 representa el borde ambiguo (trimap) del conjunto de datos Oxford-IIIT Pet. Estos píxeles no pertenecen ni al fondo ni a la mascota, por lo que deben excluirse tanto del cálculo de la función de pérdida como de las métricas para evitar que afecten la evaluación del modelo.

2.-En el ejemplo se obtienen:

TP = 3 (tres píxeles de mascota correctamente identificados).
FP = 1 (un píxel predicho como mascota cuando en realidad no lo era).
FN = 1 (un píxel de mascota que el modelo no detectó).

Aplicando la fórmula:

IoU= TP /
(TP+FP+FN)
	​
se obtiene:

IoU= 3 / (3+1+1) = 3/5
	​

=0.60

El píxel con valor 255 no participa en este cálculo porque corresponde a una región ignorada.



3.- Se acumulan primero los valores de la matriz de confusión (TP, FP y FN) de todas las imágenes para calcular una única métrica representativa del conjunto completo. De esta forma, las imágenes con más píxeles válidos tienen un peso proporcional y se evita que todas las imágenes influyan por igual, independientemente de su tamaño o cantidad de píxeles útiles.